In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import model as md


In [ ]:

def fisher_MS(x, lam):
    N = len(x)
    return ((lam - 1)**2 + 1/N)*np.outer(x, x) + np.eye(N)/N


def power_iteration(M, x_hat, iterations=50, tol=1e-7):
        for _ in range(iterations):
            x_new = M @ x_hat   # single mat-vec, O(N²)
            x_new /= np.linalg.norm(x_new)
            if np.linalg.norm(x_new - x_hat) < tol:
                break
            x_hat = x_new
        return x_hat

def x_fisher(N, rho, lambda1, lambda2, mu, method, CV=False):
    Y1, Y2, x1, x2 = md.spiked_gaussian_matrix_with_correlated_spikes(N, lambda1, lambda2, rho)

    theta1 = np.ones(N)
    theta1 = power_iteration(Y1, theta1)

    F1 = fisher_MS(theta1, lambda1)
    A = Y2 - np.eye(N)/N + 2*mu*F1

    thetaA = np.zeros(N)
    thetaB = np.zeros(N)

    if method in ("A", "both"):
        thetaA = np.ones(N)
        thetaA = power_iteration(A, thetaA)

    if method in ("B", "both"):
        b = 2*mu * F1 @ theta1
        thetaB = np.linalg.solve(A, b)

    return theta1, thetaA, thetaB, x1, x2





def ploplot(title, ylabel, lam_vals, yA, stdA, yB, stdB, ybase=None, stdbase=None):
    fig, ax = plt.subplots(figsize=(8, 5))

    ax.errorbar(lam_vals, yA, yerr=stdA, fmt='o-', capsize=4,
                color='steelblue', label='Method A', linewidth=1.5, markersize=5)
    ax.errorbar(lam_vals, yB, yerr=stdB, fmt='s-', capsize=4,
                color='tomato', label='Method B', linewidth=1.5, markersize=5)

    if ybase is not None:
        ax.errorbar(lam_vals, ybase, yerr=stdbase, fmt='D--', capsize=4,
                    color='gray', label=r'Step1 $\theta_1$', linewidth=1.5, markersize=5)

    ax.axvline(x=1, color='black', linestyle=':', linewidth=1, label=r'$\lambda=1$')
    ax.set_xlabel(r'$\lambda_1$', fontsize=13)
    ax.set_ylabel(ylabel, fontsize=13)
    ax.set_title(title, fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()



In [ ]:

N = 5000
lambda2 = 2
lambda1 = 3
mu = 1
MU = np.linspace(0.1, 100, 20)
rho = 0.2
LAM1 = np.linspace(0.1, 5, 20)
M = 10

results = {
    "A": {"over1": [], "over2": [], "std1": [], "std2": []},
    "B": {"over1": [], "over2": [], "std1": [], "std2": []},
    "step1": {"over1": [], "std1": []},
}

for mu in MU:
    mA1, mA2, mB1, mB2, mstep1 = [], [], [], [], []

    for _ in range(M):
        theta1, thetaA, thetaB, x1, x2 = x_fisher(N, rho, lambda1, lambda2, mu, "both")

        mA1.append(abs(np.dot(thetaA, x1)))
        mA2.append(abs(np.dot(thetaA, x2)))
        mB1.append(abs(np.dot(thetaB, x1)))
        mB2.append(abs(np.dot(thetaB, x2)))
        mstep1.append(abs(np.dot(theta1, x1)))

    for key, arr in zip(["over1", "over2"], [mA1, mA2]):
        results["A"][key].append(np.mean(arr))
        results["A"][f"std{key[-1]}"].append(np.std(arr) / np.sqrt(M))

    for key, arr in zip(["over1", "over2"], [mB1, mB2]):
        results["B"][key].append(np.mean(arr))
        results["B"][f"std{key[-1]}"].append(np.std(arr) / np.sqrt(M))

    results["step1"]["over1"].append(np.mean(mstep1))
    results["step1"]["std1"].append(np.std(mstep1) / np.sqrt(M))

ploplot(
    title=r'Overlap with $x_1$ vs $\lambda_1$',
    ylabel=r'$|\langle \hat{\theta}, x_1 \rangle|$',
    lam_vals=LAM1,
    yA=results["A"]["over1"], stdA=results["A"]["std1"],
    yB=results["B"]["over1"], stdB=results["B"]["std1"],
    ybase=results["step1"]["over1"], stdbase=results["step1"]["std1"],
)

ploplot(
    title=r'Overlap with $x_2$ vs $\lambda_1$',
    ylabel=r'$|\langle \hat{\theta}, x_2 \rangle|$',
    lam_vals=LAM1,
    yA=results["A"]["over2"], stdA=results["A"]["std2"],
    yB=results["B"]["over2"], stdB=results["B"]["std2"],
)

In [ ]:
def run_experiment_2D(vary_param_x, vary_values_x, vary_param_y, vary_values_y, 
                       M, n, k1, k2, rho, alpha, method, methodF = "A", all_overlap = True, plot = True):
    
    Z11    = np.zeros((len(vary_values_y), len(vary_values_x)))
    Z22   = np.zeros((len(vary_values_y), len(vary_values_x)))
    Z12 = np.zeros((len(vary_values_y), len(vary_values_x)))
    Z21 = np.zeros((len(vary_values_y), len(vary_values_x)))

    for i, val_y in enumerate(vary_values_y):
        for j, val_x in enumerate(vary_values_x):
            params = {'n': n, 'k1': k1, 'k2': k2, 'rho': rho, 'alpha': alpha}
            params[vary_param_x] = val_x
            params[vary_param_y] = val_y

            over11, over22, over12, over21 = [], [], [], []
            for _ in range(M):
                Y1, Y2, x1, x2 = md.spiked_gaussian_matrix_with_correlated_spikes(params['n'], params['k1'],  params['k2'], params['rho'])
                D = params['alpha'] * Y1 + np.sqrt(1 - params['alpha']**2)*Y2
                D, x1, x2 = md.spec(params['n'], params['k1'], params['k2'], 
                                      params['rho'], params['alpha'])
                if method == "naive": 
                    m11, m22, m12, m21 = md.overlap(D, x1, x2, all_overlap)
                elif method == "fisher":
                    theta1, thetaA, thetaB, x1, x2 = x_fisher(N, rho, lambda1, lambda2, mu, method = methodF)
                    m11 = abs(np.dot(thetaA,x1))
                    m12 = abs(np.dot(thetaA,x2))
                    m21 = abs(np.dot(thetaB,x1))
                    m22 = abs(np.dot(thetaB,x2))
                else: 
                    m11, m12, m22, m21 = 0  
                     
                over11.append(m11)
                over22.append(m22)
                over12.append(m12)
                over21.append(m21)

            Z11[i, j]    = np.mean(over11)
            Z21[i, j] = np.mean(over21)
            if all_overlap:
                Z22[i, j]    = np.mean(over22)
                Z12[i, j] = np.mean(over12)
                
    if plot : 
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.flatten()
        titles = ['vA.x1', 'vB.x2', 'vA.x2', 'vB.x1']
        Zs = [Z11, Z22, Z12, Z21]

        for ax, Z, title in zip(axes, Zs, titles):
            im = ax.imshow(Z, origin='lower', aspect='auto', cmap='viridis',
                        extent=[vary_values_x[0], vary_values_x[-1], 
                                vary_values_y[0], vary_values_y[-1]],vmin=0, vmax=1)
            fig.colorbar(im, ax=ax, label='Overlap')
            ax.set_xlabel(vary_param_x, fontsize=12)
            ax.set_ylabel(vary_param_y, fontsize=12)
            ax.set_title(title, fontsize=14)

        plt.tight_layout()
        plt.show()

            # fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            # axes = axes.flatten()
            # titles = ['max over x1', 'max over x2']
            # Zs = [np.maximum(Z11,Z12), np.maximum(Z22,Z21)]

        # for ax, Z, title in zip(axes, Zs, titles):
        #     im = ax.imshow(Z, origin='lower', aspect='auto', cmap='viridis',
        #                 extent=[vary_values_x[0], vary_values_x[-1], 
        #                         vary_values_y[0], vary_values_y[-1]],vmin=0, vmax=1)
        #     fig.colorbar(im, ax=ax, label='Overlap')
        #     ax.set_xlabel(vary_param_x, fontsize=12)
        #     ax.set_ylabel(vary_param_y, fontsize=12)
        #     ax.set_title(title, fontsize=14)
        # plot f(k1) = sqrt( (k2/k1)^2 / (1 + (k2/k1)^2) )
        #    k1_vals = vary_values_x[vary_values_x > 1]
        #    k2=2
            
        #   f_vals = np.sqrt((k2 / k1_vals)**2 / (1 + (k2 / k1_vals)**2))
        #    ax.plot(k1_vals, f_vals, color='red', linewidth=2, label=r'$f(k_1)$')
        #    ax.legend(fontsize=12)
        # plt.tight_layout()
        # plt.show()
    

    return Z11, Z22, Z12, Z21

In [ ]:
#varry: rho, alpha, k1, k1-k2
M=10 #nb of simulation for averaging (precision)
n=100 #size of signal
k1= 5
k2= 2
alpha = np.sqrt(0.5) #Y1 plus imp que Y2
rho = 1

RHO = np.linspace(0,1,30)
K1 = np.linspace(0,5,30)

K2 = np.linspace(0,5,30)
ALPHA = np.linspace(0,1,30)

ZA1, ZB2, ZA2, ZB1 = run_experiment_2D(
    vary_param_x='k1', vary_values_x=K1,
    vary_param_y='rho',  vary_values_y=RHO,
    M=M, n=n, k1=k1, k2=k2, rho=rho, alpha=alpha,method = "fisher", methodF = "A",all_overlap= False, plot = True
)

